In [ ]:
# X4-A — Direct token/logit forensic capture for Veena kavya drift.
# READ-ONLY: no production kernel touched, no model weights modified.
import json, os, sys, time, hashlib, subprocess, io, wave
print('=== X4A START', time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))

# 1) Env fingerprint (all values captured verbatim) ---------------------------
import torch, transformers
env = {
    'torch': torch.__version__,
    'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(),
    'transformers': transformers.__version__,
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_cap': torch.cuda.get_device_capability(0) if torch.cuda.is_available() else None,
    'gpu_count': torch.cuda.device_count(),
    'bf16_supported': torch.cuda.is_bf16_supported() if torch.cuda.is_available() else None,
    'tf32_matmul': torch.backends.cuda.matmul.allow_tf32,
    'tf32_cudnn': torch.backends.cudnn.allow_tf32,
    'matmul_precision': None,
    'deterministic': torch.are_deterministic_algorithms_enabled(),
}
try: env['matmul_precision'] = torch.get_float32_matmul_precision()
except Exception: pass
print('ENV:', json.dumps(env, indent=2))

# 2) Install snac==1.0.0 if not present ---------------------------------------
try:
    import snac
except Exception:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'snac==1.0.0', 'huggingface_hub'])
    import snac
env['snac'] = getattr(snac, '__version__', 'unknown')
print('snac:', env['snac'])

# 3) Load Veena BF16 exactly like production ----------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer
print('Loading Veena...')
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained('maya-research/Veena')
model = AutoModelForCausalLM.from_pretrained('maya-research/Veena', torch_dtype=torch.bfloat16, device_map='cuda')
model.eval()
print(f'Loaded Veena in {time.time()-t0:.1f}s')
env['model_config_dtype'] = str(model.dtype)
env['model_num_params'] = sum(p.numel() for p in model.parameters())
# Model revision — the HF hub sometimes exposes commit hash in .config._commit_hash
env['model_commit_hash'] = getattr(model.config, '_commit_hash', None) or getattr(model, '_commit_hash', None)
env['attn_impl'] = str(getattr(model.config, '_attn_implementation', None))
print('model dtype:', env['model_config_dtype'], '| attn:', env['attn_impl'], '| commit:', env['model_commit_hash'])

# 4) Load SNAC with production monkey-patches --------------------------------
from snac import SNAC
from huggingface_hub import hf_hub_download
import types as _types
snac_cfg_path = hf_hub_download(repo_id='hubertsiuzdak/snac_24khz', filename='config.json')
snac_wts_path = hf_hub_download(repo_id='hubertsiuzdak/snac_24khz', filename='pytorch_model.bin')
with open(snac_cfg_path) as f: snac_cfg = json.load(f)
snac_model = SNAC(**snac_cfg)
def _strip_attn(seq):
    return torch.nn.Sequential(*[m for m in seq.children() if type(m).__name__ != 'LocalMHA'])
snac_model.encoder.block = _strip_attn(snac_model.encoder.block)
snac_model.decoder.model = _strip_attn(snac_model.decoder.model)
snac_state = torch.load(snac_wts_path, map_location='cpu', weights_only=False)
snac_model.load_state_dict(snac_state)
snac_model.eval()
snac_model = snac_model.to('cuda')
def _snac_decode_compat(self, codes):
    z_q = 0
    for quantizer, code in zip(self.quantizer.quantizers, codes):
        z_q_i = quantizer.decode_code(code)
        z_q_i = quantizer.out_proj(z_q_i)
        if quantizer.stride > 1:
            z_q_i = z_q_i.repeat_interleave(quantizer.stride, dim=-1)
        z_q = z_q + z_q_i
    return self.decoder(z_q)
snac_model.decode = _types.MethodType(_snac_decode_compat, snac_model)
import snac.layers as _snac_layers
def _snake_plain(x, alpha):
    shape = x.shape
    x = x.reshape(shape[0], shape[1], -1)
    x = x + (alpha + 1e-9).reciprocal() * torch.sin(alpha * x).pow(2)
    x = x.reshape(shape)
    return x
_snac_layers.snake = _snake_plain
print('SNAC loaded with production patches (LocalMHA stripped, snake→Python, decode() patched)')

# 5) Constants (must match production) ---------------------------------------
START_OF_HUMAN = 128259
END_OF_HUMAN   = 128260
START_OF_AI    = 128261
END_OF_AI      = 128262
START_OF_SPEECH= 128257
END_OF_SPEECH  = 128258
AUDIO_BASE     = 128266
CODEBOOK_SIZE  = 4096
TOKENS_PER_FRAME = 7
SNAC_MIN = AUDIO_BASE
SNAC_MAX = AUDIO_BASE + TOKENS_PER_FRAME * CODEBOOK_SIZE - 1
SR = 24000

# 6) Speaker embedding analysis ---------------------------------------------
SPEAKERS = ['kavya','apsara','agastya','vinaya','maitri','charu','ishana','kyra','mohini','varun','soumya']
spk_ids = {}
for s in SPEAKERS:
    tok = tokenizer.encode(f'<spk_{s}>', add_special_tokens=False)
    spk_ids[s] = tok
print('speaker token IDs:', spk_ids)

emb = model.get_input_embeddings().weight.detach()  # shape [V, H]
print('embed dtype:', emb.dtype, 'shape:', tuple(emb.shape))
spk_analysis = {'ids': spk_ids, 'norms': {}, 'cos_to_kavya': {}, 'cos_matrix': {}}
for s in SPEAKERS:
    if len(spk_ids[s]) != 1:
        spk_analysis['norms'][s] = None
        continue
    v = emb[spk_ids[s][0]].float()
    spk_analysis['norms'][s] = float(v.norm().item())
kv = emb[spk_ids['kavya'][0]].float()
kv_n = kv / (kv.norm() + 1e-9)
for s in SPEAKERS:
    if len(spk_ids[s]) != 1: continue
    v = emb[spk_ids[s][0]].float()
    v_n = v / (v.norm() + 1e-9)
    spk_analysis['cos_to_kavya'][s] = float((kv_n * v_n).sum().item())
# full pairwise matrix
for a in SPEAKERS:
    if len(spk_ids[a]) != 1: continue
    va = emb[spk_ids[a][0]].float(); va_n = va / (va.norm() + 1e-9)
    row = {}
    for b in SPEAKERS:
        if len(spk_ids[b]) != 1: continue
        vb = emb[spk_ids[b][0]].float(); vb_n = vb / (vb.norm() + 1e-9)
        row[b] = float((va_n * vb_n).sum().item())
    spk_analysis['cos_matrix'][a] = row
print('cos-to-kavya:', json.dumps(spk_analysis['cos_to_kavya'], indent=2))
print('norms:', json.dumps(spk_analysis['norms'], indent=2))

# 7) Corpus (subset of X3 for time; includes all required + minimal-pair) ----
CORPUS = [
    ('drift',  'Aapke bank se transfer complete ho gaya hai.'),
    ('drift',  'Sir kya aap UPI se payment karna prefer karenge?'),
    ('drift',  'Total outstanding 24,568 rupees hai as of aaj.'),
    ('drift',  'Principal amount 15,000 rupees baaki hai.'),
    ('stable', 'Namaste sir, main Kavya bol rahi hoon Rajat Finance se.'),
    ('stable', 'Namaste madam, main Kavya bol rahi hoon.'),
    ('stable', 'Namaste sir aap kaise hain aaj?'),
    ('stable', 'Dhanyavaad sir, aapke response ka intezaar rahega.'),
    ('stable', 'Dhanyavaad, aapki payment successful ho gayi hai.'),
    ('stable', 'Sir, aapke account par 12,500 rupees ka outstanding hai.'),
    ('stable', 'Aap kaise hain?'),
    ('stable', 'Kripa karke wait karein.'),
    ('english','Good morning, this is a test message.'),
    ('english','Your account balance is one thousand five hundred rupees.'),
    ('num_heavy','9,876,543 rupees ka total amount pending hai.'),
    ('min_pair','Total outstanding hai as of aaj.'),
    ('min_pair','Total outstanding amount check kar rahi hoon.'),
    ('min_pair','Aapke bank se successful transaction huwa hai.'),
    ('min_pair','Bank transfer ho chuka hai sir, dhyaan dijiye.'),
]
print(f'Corpus: {len(CORPUS)} texts')

# 8) Helper: F0 autocorrelation on decoded PCM ------------------------------
import numpy as np
def f0_frame(frame, sr=SR, fmin=70, fmax=400):
    frame = frame.astype(np.float32) - frame.mean()
    if np.sqrt((frame*frame).mean()) < 300: return 0.0
    corr = np.correlate(frame, frame, mode='full')
    corr = corr[len(corr)//2:]
    corr = corr / (corr[0] + 1e-9)
    lag_min = sr // fmax; lag_max = sr // fmin
    seg = corr[lag_min:lag_max]
    if len(seg)==0: return 0.0
    peak = int(np.argmax(seg)) + lag_min
    if corr[peak] < 0.30: return 0.0
    return sr / peak

def analyze(pcm_i16):
    n = len(pcm_i16); WIN=480; HOP=240
    if n < WIN: return {'class':'silent','median_f0':None,'p05_f0':None,'f0':[]}
    nf = 1 + (n - WIN)//HOP
    x = pcm_i16.astype(np.float32)
    f0s=[]; f0_traj=[]
    for i in range(nf):
        f = f0_frame(x[i*HOP:i*HOP+WIN])
        f0_traj.append(round(f,1))
        if f>0: f0s.append(f)
    if not f0s: return {'class':'silent','median_f0':None,'p05_f0':None,'f0':f0_traj}
    a=np.array(f0s); med=float(np.median(a)); p05=float(np.percentile(a,5))
    cls = 'female-kavya' if (med>=180 and p05>=140) else ('male-drift' if (med<165 or p05<120) else 'ambiguous')
    return {'class':cls,'median_f0':med,'p05_f0':p05,'f0':f0_traj}

def decode_snac_audio(audio_ids_int):
    if len(audio_ids_int) < 7: return np.zeros(0, dtype=np.int16)
    n_frames = len(audio_ids_int) // 7
    audio_ids_int = audio_ids_int[:n_frames*7]
    codes_l0 = []  # coarse
    codes_l1 = []  # mid
    codes_l2 = []  # fine
    for f in range(n_frames):
        b = f*7
        codes_l0.append(audio_ids_int[b+0])
        codes_l1.append(audio_ids_int[b+1]); codes_l1.append(audio_ids_int[b+4])
        codes_l2.append(audio_ids_int[b+2]); codes_l2.append(audio_ids_int[b+3])
        codes_l2.append(audio_ids_int[b+5]); codes_l2.append(audio_ids_int[b+6])
    with torch.no_grad():
        c0 = torch.tensor([codes_l0], device='cuda', dtype=torch.long)
        c1 = torch.tensor([codes_l1], device='cuda', dtype=torch.long)
        c2 = torch.tensor([codes_l2], device='cuda', dtype=torch.long)
        wav = snac_model.decode([c0, c1, c2]).squeeze().float().cpu().numpy()
    pcm = (np.clip(wav, -1.0, 1.0) * 32767).astype(np.int16)
    return pcm

def audio_ids_from_generated(seq_ids, prompt_len):
    """Extract raw audio-token subsequence (already 0-indexed codebook vals)."""
    tail = seq_ids[prompt_len:]
    aud = []
    pos = 0
    for tid in tail:
        if SNAC_MIN <= tid <= SNAC_MAX:
            p = pos % TOKENS_PER_FRAME
            v = tid - AUDIO_BASE - p * CODEBOOK_SIZE
            aud.append(max(0, min(v, CODEBOOK_SIZE-1)))
            pos += 1
    return aud

# 9) Core: generate with top-5 capture --------------------------------------
def build_input(text, speaker='kavya'):
    prompt = f'<spk_{speaker}> {text}'
    pids = tokenizer.encode(prompt, add_special_tokens=False)
    ids = [START_OF_HUMAN, *pids, END_OF_HUMAN, START_OF_AI, START_OF_SPEECH]
    return torch.tensor([ids], device='cuda'), ids

def capture_generation(text, seed=42, max_new=None):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    input_ids, ids_list = build_input(text)
    prompt_len = input_ids.shape[1]
    if max_new is None:
        max_new = min(int(len(text)*1.3) * TOKENS_PER_FRAME + 21, 700)
    with torch.no_grad():
        out = model.generate(
            input_ids,
            max_new_tokens=max_new,
            do_sample=True,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.05,
            pad_token_id=(tokenizer.pad_token_id if tokenizer.pad_token_id is not None else END_OF_SPEECH),
            eos_token_id=[END_OF_SPEECH, END_OF_AI],
            output_scores=True,
            return_dict_in_generate=True,
        )
    seq = out.sequences[0].tolist()
    gen_tail = seq[prompt_len:]
    scores = out.scores  # list of tensors [1, vocab] per step (post-processor logits)
    # top-5 per position
    per_pos = []
    K = 5
    for i, s in enumerate(scores):
        logits = s[0].float()
        probs = torch.softmax(logits, dim=-1)
        topv, topi = torch.topk(logits, K)
        chosen = int(gen_tail[i]) if i < len(gen_tail) else -1
        # top-2 margin on RAW logits (post processors: temperature already scaled by 1/T)
        # note: HF's process_logits applies temperature-scaling on the logits before softmax
        top1_id = int(topi[0].item()); top2_id = int(topi[1].item())
        top1_logit = float(topv[0].item()); top2_logit = float(topv[1].item())
        margin = top1_logit - top2_logit
        # is chosen == argmax?
        chosen_rank = -1
        for r, tid in enumerate(topi.tolist()):
            if tid == chosen: chosen_rank = r; break
        per_pos.append({
            'pos': i,
            'chosen': chosen,
            'chosen_rank_in_top5': chosen_rank,
            'chosen_prob': float(probs[chosen].item()) if 0 <= chosen < probs.shape[0] else None,
            'top5_ids': [int(x) for x in topi.tolist()],
            'top5_logits': [float(x) for x in topv.tolist()],
            'top5_probs': [float(probs[int(t)].item()) for t in topi.tolist()],
            'top1_top2_margin': margin,
            'is_audio': (SNAC_MIN <= chosen <= SNAC_MAX),
            'codebook_pos': None,
            'codebook_val': None,
        })
        # codebook decoding if audio
        if SNAC_MIN <= chosen <= SNAC_MAX:
            # count only audio tokens up to & including this one
            audio_count = sum(1 for p in per_pos if p['is_audio'])
            cb_pos = (audio_count-1) % TOKENS_PER_FRAME
            cb_val = chosen - AUDIO_BASE - cb_pos * CODEBOOK_SIZE
            per_pos[-1]['codebook_pos'] = cb_pos
            per_pos[-1]['codebook_val'] = max(0, min(cb_val, CODEBOOK_SIZE-1))
    return {'seq': seq, 'prompt_len': prompt_len, 'per_pos': per_pos}

# 10) Run main corpus ------------------------------------------------------
records = []
for idx, (bucket, text) in enumerate(CORPUS):
    t0 = time.time()
    cap = capture_generation(text, seed=42)
    latency = time.time() - t0
    aud_vals = audio_ids_from_generated(cap['seq'], cap['prompt_len'])
    pcm = decode_snac_audio(aud_vals) if len(aud_vals) >= 21 else np.zeros(0, dtype=np.int16)
    ana = analyze(pcm)
    # first-21 detailed capture
    first_21_audio_positions = [p for p in cap['per_pos'] if p['is_audio']][:21]
    rec = {
        'idx': idx, 'bucket': bucket, 'text': text,
        'prompt_len': cap['prompt_len'],
        'n_gen': len(cap['per_pos']),
        'n_audio': sum(1 for p in cap['per_pos'] if p['is_audio']),
        'first_10_gen_positions': cap['per_pos'][:10],  # includes any non-audio prefix
        'first_21_audio_positions': first_21_audio_positions,
        'audio_ids_full': aud_vals,
        'audio_sha16': hashlib.sha256(bytes(pcm)).hexdigest()[:16],
        'audio_seconds': len(pcm)/SR,
        'class': ana['class'], 'median_f0': ana['median_f0'], 'p05_f0': ana['p05_f0'],
        'f0_trajectory': ana['f0'],
        'latency_s': latency,
    }
    records.append(rec)
    top21_margins = [round(p['top1_top2_margin'],3) for p in first_21_audio_positions]
    top21_ranks = [p['chosen_rank_in_top5'] for p in first_21_audio_positions]
    print(f'[{idx:2d}][{bucket:9s}] cls={ana["class"]:12s} med={ana["median_f0"]} p05={ana["p05_f0"]} '
          f'n_aud={rec["n_audio"]:3d} sha16={rec["audio_sha16"]} lat={latency:.1f}s '
          f'first21_margins_med={round(float(np.median(top21_margins)),3) if top21_margins else "na"} '
          f'| {text[:45]!r}')

# 11) Determinism: 3 drift + 3 stable × 3 reps -----------------------------
print('\n=== DETERMINISM (3 drift × 3 rep + 3 stable × 3 rep) ===')
DET_IDX = [2, 3, 8, 6, 10, 0]  # 2:drift 24568, 3:drift Principal, 8:stable Dhanyavaad payment, 6:stable Namaste sir, 10:stable Aap kaise, 0:drift bank
det = []
for di in DET_IDX:
    text = CORPUS[di][1]
    reps = []
    for r in range(3):
        cap = capture_generation(text, seed=42)
        aud_vals = audio_ids_from_generated(cap['seq'], cap['prompt_len'])
        pcm = decode_snac_audio(aud_vals) if len(aud_vals)>=21 else np.zeros(0, dtype=np.int16)
        ana = analyze(pcm)
        seq_first21_audio = [p['chosen'] for p in cap['per_pos'] if p['is_audio']][:21]
        reps.append({
            'rep': r, 'audio_sha16': hashlib.sha256(bytes(pcm)).hexdigest()[:16],
            'first21_audio_ids': seq_first21_audio,
            'class': ana['class'], 'median_f0': ana['median_f0'], 'p05_f0': ana['p05_f0'],
        })
    unique_sha = len({r['audio_sha16'] for r in reps})
    unique_first21 = len({tuple(r['first21_audio_ids']) for r in reps})
    print(f'  [{di:2d}] unique_sha16={unique_sha} unique_first21={unique_first21} classes={[r["class"] for r in reps]} | {text[:40]!r}')
    det.append({'idx': di, 'text': text, 'reps': reps,
                'unique_sha16': unique_sha, 'unique_first21_audio': unique_first21})

# 12) Write JSON output ----------------------------------------------------
OUT = '/kaggle/working/x4a_results.json'
with open(OUT, 'w') as f:
    json.dump({
        'env': env,
        'speaker_embedding_analysis': spk_analysis,
        'corpus': CORPUS,
        'records': records,
        'determinism': det,
    }, f, indent=1, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))
print(f'WROTE {OUT}, size={os.path.getsize(OUT)} bytes')
print('=== X4A DONE', time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()))
